In [1]:
# Install openrouteservice for first time users
# !pip install openrouteservice

In [2]:
import openrouteservice
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
# Load in rental data
rental = pd.read_csv("rea_data/domain/Data/vic_rentals_all_cleaned.csv")
rental_coords = rental[["listing_id", "lon","lat"]]
rental_coords.head()

,listing_id,lon,lat
0,16782629,144.87091,-37.830982
1,17471867,144.86755,-37.826218
2,17721851,144.86632,-37.831226
3,17725855,144.86768,-37.827423
4,17745057,144.86790,-37.826270


In [4]:
# Load in school data
school = pd.read_csv("data\dv402-SchoolLocations2025.csv")
school_coords = school[["School_No", "X","Y"]]
school_coords.head()

,School_No,X,Y
0,20,145.066978,-37.690178
1,25,144.952883,-37.805971
2,26,144.997001,-37.859365
3,28,143.831558,-37.559711
4,29,143.847147,-37.564397


In [5]:
# Define Haversine distance function
def haversine_vec(lon1, lat1, lon2, lat2):
    """
    Compute distance (meters) between two points using vectorised Haversine formula
    """
    lon1_rad = np.radians(lon1)[:, np.newaxis]
    lat1_rad = np.radians(lat1)[:, np.newaxis]

    lon2_rad = np.radians(lon2)[np.newaxis, :]
    lat2_rad = np.radians(lat2)[np.newaxis, :]

    earth_mean_radius = 6371000
    phi1, phi2 = lat1_rad, lat2_rad
    dphi1 = lat2_rad - lat1_rad
    dlambda = lon2_rad - lon1_rad
    
    a = np.sin(dphi1/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return earth_mean_radius * c

In [6]:
# Find top 5 closest schools for each rental
dist_matrix = haversine_vec(rental_coords["lon"].values, rental_coords["lat"].values, school_coords["X"].values, school_coords["Y"].values)

top5_indices = np.argsort(dist_matrix, axis=1)[:,:5]

rental_coords["top5_idx"] = list(top5_indices)
rental_coords["top5_tuple"] = [tuple(int(i) for i in sorted(x)) for x in top5_indices]

C:\Users\chinj\AppData\Local\Temp\ipykernel_2896\4010516909.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top5_idx"] = list(top5_indices)
C:\Users\chinj\AppData\Local\Temp\ipykernel_2896\4010516909.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top5_tuple"] = [tuple(int(i) for i in sorted(x)) for x in top5_indices]


In [7]:
# Group results by their top 5 closest schools
groups = rental_coords.groupby("top5_tuple")["listing_id"].apply(list).reset_index()
groups["num_rentals"] = groups["listing_id"].apply(len)

In [8]:
# API key for OpenRouteService (ORS); hidden here for privacy
API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImEyMGE4NjZkNjYyZDQ4NDdiN2I4ODJhOWQ1ODM0YThhIiwiaCI6Im11cm11cjY0In0=" # Replace NA with your actual key when running

# Initialize ORS client with the API key
client = openrouteservice.Client(key = API_KEY)

In [9]:
# Batch rentals to stay under API route limit
top_n_schools = 5
max_route_per_batch = 3500

batches = []
current_batch = []
current_batch_rentals = 0

# Loop over each group of rentals that share the same top 5 schools
for _, row in groups.iterrows():
    group_name = row["top5_tuple"]
    rentals = row["listing_id"]
    group_size = len(rentals)

    start = 0
    # Split group into chunks so that adding them won't exceed ORS route limit
    while start < group_size:
        # Estimate number of rentals allowed to add to current batch without exceeding ORS route limit
        num_groups_if_added = len(current_batch) + 1
        max_rentals_for_this_chunk = max(
            (max_route_per_batch // (num_groups_if_added * top_n_schools)) - current_batch_rentals,
            1
        )

        # Determine slice of rentals for this chunk
        end = min(start + max_rentals_for_this_chunk, group_size)
        chunk = rentals[start:end]

        # Compute estimated number of routes if this chunk is added
        num_origins = current_batch_rentals + len(chunk)
        num_destinations = num_groups_if_added * top_n_schools
        estimated_routes = num_origins * num_destinations

        # Start new batch if adding this chunk exceeds ORS route limit
        if estimated_routes >= max_route_per_batch: 
            if current_batch:
                batches.append(current_batch)
            current_batch = [(group_name, chunk)]     
            current_batch_rentals = len(chunk)
        else:
            # Add chunk to batch otherwise
            current_batch.append((group_name, chunk))
            current_batch_rentals += len(chunk)
        start = end

# Append last batch if there are any remaining rentals
if current_batch:
        batches.append(current_batch)

print(f"Created {len(batches)} batches with full ORS route limit logic.")

Created 240 batches with full ORS route limit logic.


In [10]:
# Print and check summary of batches
for i, batch in enumerate(batches):
    num_groups = len(batch)
    num_rentals = sum(len(rentals) for _, rentals in batch)
    print(f"Batch {i+1}: {num_groups} groups, {num_rentals} rentals")

Batch 1: 16 groups, 42 rentals
Batch 2: 6 groups, 116 rentals
Batch 3: 6 groups, 116 rentals
Batch 4: 6 groups, 116 rentals
Batch 5: 9 groups, 69 rentals
Batch 6: 10 groups, 64 rentals
Batch 7: 17 groups, 41 rentals
Batch 8: 14 groups, 49 rentals
Batch 9: 16 groups, 43 rentals
Batch 10: 8 groups, 83 rentals
Batch 11: 16 groups, 43 rentals
Batch 12: 12 groups, 58 rentals
Batch 13: 17 groups, 39 rentals
Batch 14: 15 groups, 44 rentals
Batch 15: 13 groups, 49 rentals
Batch 16: 18 groups, 38 rentals
Batch 17: 16 groups, 41 rentals
Batch 18: 16 groups, 43 rentals
Batch 19: 10 groups, 64 rentals
Batch 20: 12 groups, 58 rentals
Batch 21: 17 groups, 41 rentals
Batch 22: 12 groups, 57 rentals
Batch 23: 18 groups, 36 rentals
Batch 24: 19 groups, 36 rentals
Batch 25: 20 groups, 34 rentals
Batch 26: 16 groups, 43 rentals
Batch 27: 15 groups, 43 rentals
Batch 28: 12 groups, 57 rentals
Batch 29: 9 groups, 70 rentals
Batch 30: 14 groups, 46 rentals
Batch 31: 19 groups, 36 rentals
Batch 32: 12 groups,

In [11]:
# Call ORS to get distance to closest schools
all_rental_ids = []
all_distances = []
all_school_ids = []

for batch in tqdm(batches, desc="Processing batches"):
    batch_rentals = [r for _, rentals in batch for r in rentals]
    batch_coords = rental_coords.set_index("listing_id").loc[batch_rentals][["lon", "lat"]].values.tolist()

    batch_school_indices = [
        rental_coords.loc[rental_coords["listing_id"]==r, "top5_idx"].values[0] for r in batch_rentals
    ]

    unique_school_indices = list(set(idx for indices in batch_school_indices for idx in indices))

    origins = batch_coords
    destinations = school_coords.loc[school_coords.index.isin(unique_school_indices)][["X", "Y"]].values.tolist()
    dest_ids_map = {idx: school_coords["School_No"].iloc[idx] for idx in unique_school_indices}
   
    school_pos_map = {idx: i for i, idx in enumerate(unique_school_indices)}

    num_origins = len(origins)
    num_destinations = len(destinations)
    num_routes = num_origins * num_destinations
    print(f"Checking batch → {num_origins} origins × {num_destinations} destinations = {num_routes} routes")

    break

    # Call ORS distance matrix
    matrix = client.distance_matrix(
        locations = origins + destinations,
        profile = "driving-car",
        metrics = ["distance"],
        sources = list(range(len(origins))),
        destinations = list(range(len(origins), len(origins)+len(destinations)))
    )

    # Extract distances and school IDs for top 5 schools
    for i, rental in enumerate(batch_rentals):
        top_indices = batch_school_indices[i]
        dest_positions = [school_pos_map[idx] for idx in top_indices]
        rental_distances = [matrix["distances"][i][pos] for pos in dest_positions]
        rental_school_ids = [dest_ids_map[idx] for idx in top_indices]

        sorted_idx = np.argsort(rental_distances)
        top_n_distances = [rental_distances[k] for k in sorted_idx]
        top_n_ids = [rental_school_ids[k] for k in sorted_idx]

        all_rental_ids.append(batch_rentals[i])
        all_distances.append(top_n_distances)
        all_school_ids.append(top_n_ids)

Processing batches:   0%|          | 0/240 [00:00<?, ?it/s]

Checking batch → 42 origins × 25 destinations = 1050 routes


In [12]:
# Save ORS distances for schools into dataframe
df = pd.DataFrame({
    "rental_id": all_rental_ids,
    "top_distances": all_distances,
    "top_school_ids": all_school_ids
})

df.head()

,rental_id,top_distances,top_school_ids


In [13]:
# Save final CSV
df.to_csv("rentals_distances_to_schools.csv", index=False)